In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertTokenizer
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import TrainingArguments
from transformers import BertTokenizer, BertForSequenceClassification

In [2]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim=512, num_heads=8,dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        assert self.head_dim * num_heads == embed_dim
        self.q_linear = nn.Linear(embed_dim, embed_dim)
        self.k_linear = nn.Linear(embed_dim, embed_dim)
        self.v_linear = nn.Linear(embed_dim, embed_dim)
        self.fc_out = nn.Linear(embed_dim, embed_dim)
        self.attn_dropout = nn.Dropout(dropout)


    def forward(self, x, attention_mask=None):
        B, T, C = x.shape
        Q = self.q_linear(x)
        K = self.k_linear(x)
        V = self.v_linear(x)
        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.fc_out(out)

In [3]:
class FeedForward(nn.Module):
    def __init__(self, embed_dim=512, hidden_dim=2048, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.fc2(self.dropout(F.relu(self.fc1(x))))

In [4]:
class Encoder(nn.Module):
    def __init__(self, embed_dim=512, num_heads=8, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadSelfAttention(embed_dim, num_heads)
        self.ff = FeedForward(embed_dim)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)


    def forward(self, x, attention_mask=None):
        attn_out = self.attn(x)
        x = self.norm1(x + attn_out)
        ff_out = self.ff(x)
        x = self.norm2(x + ff_out)
        return x

In [5]:

class Bert(nn.Module):
    def __init__(self, vocab_size, embed_dim=512, num_heads=8, num_layers=2, max_len=512, dropout=0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(max_len, embed_dim)
        self.token_type_embedding = nn.Embedding(2, embed_dim)
        self.layers = nn.ModuleList([
            Encoder(embed_dim, num_heads)
            for _ in range(num_layers)
        ])
        self.embed_dropout = nn.Dropout(p=0.3)
        self.pooler = nn.Linear(embed_dim, embed_dim)
        self.classifier = nn.Linear(embed_dim, 2)  

    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        B, T = input_ids.shape
        positions = torch.arange(0, T).unsqueeze(0).expand(B, T)
        token_emb = self.token_embedding(input_ids)
        pos_emb = self.position_embedding(positions)
        seg_emb = self.token_type_embedding(token_type_ids)
        x = self.embed_dropout(token_emb + pos_emb + seg_emb)
        print("Token Embedding Shape:", token_emb.shape)
        print("Positional Embedding Shape:", pos_emb.shape)
        print("Segment Embedding Shape:", seg_emb.shape)
        for i, layer in enumerate(self.layers):
            x = layer(x, attention_mask)
            print(f"Encoder Layer {i+1} Output Shape:", x.shape)
        cls_token = x[:, 0]
        pooled = torch.tanh(self.pooler(cls_token))
        logits = self.classifier(pooled)
        probs = torch.softmax(logits, dim=-1)
        print("Classifier Output Shape:", probs.shape)
        return probs

In [6]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
sentence = "You have to fight to reach your dream. You have to sacrifice and work hard for it."
tokens = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True)
attention_mask = tokens["attention_mask"]
input_ids = tokens["input_ids"]
token_type_ids = torch.zeros_like(input_ids)
print("Input IDs Shape:", input_ids.shape)
print("Attention Mask Shape:", attention_mask.shape)
print("Token Type IDs Shape:", token_type_ids.shape)
model = Bert(vocab_size=tokenizer.vocab_size)
output = model(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
print("Shape of each parameter : ")
for name, param in model.named_parameters():
    print(f"{name}: {list(param.shape)}")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Input IDs Shape: torch.Size([1, 21])
Attention Mask Shape: torch.Size([1, 21])
Token Type IDs Shape: torch.Size([1, 21])
Token Embedding Shape: torch.Size([1, 21, 512])
Positional Embedding Shape: torch.Size([1, 21, 512])
Segment Embedding Shape: torch.Size([1, 21, 512])
Encoder Layer 1 Output Shape: torch.Size([1, 21, 512])
Encoder Layer 2 Output Shape: torch.Size([1, 21, 512])
Classifier Output Shape: torch.Size([1, 2])
Shape of each parameter : 
token_embedding.weight: [30522, 512]
position_embedding.weight: [512, 512]
token_type_embedding.weight: [2, 512]
layers.0.attn.q_linear.weight: [512, 512]
layers.0.attn.q_linear.bias: [512]
layers.0.attn.k_linear.weight: [512, 512]
layers.0.attn.k_linear.bias: [512]
layers.0.attn.v_linear.weight: [512, 512]
layers.0.attn.v_linear.bias: [512]
layers.0.attn.fc_out.weight: [512, 512]
layers.0.attn.fc_out.bias: [512]
layers.0.ff.fc1.weight: [2048, 512]
layers.0.ff.fc1.bias: [2048]
layers.0.ff.fc2.weight: [512, 2048]
layers.0.ff.fc2.bias: [512]
l

In [7]:
df_all_test_public = pd.read_csv("/kaggle/input/datasets/parthupadhyay89/assignment-3/all_test_public.tsv", sep="\t")
df_all_train_public = pd.read_csv("/kaggle/input/datasets/parthupadhyay89/assignment-3/all_train.tsv", sep="\t")
df_all_validate_public = pd.read_csv("/kaggle/input/datasets/parthupadhyay89/assignment-3/all_validate.tsv", sep="\t")
df_multimodel_test_public = pd.read_csv("/kaggle/input/datasets/parthupadhyay89/assignment-3/multimodal_test_public.tsv", sep="\t")
df_multimodel_train_public = pd.read_csv("/kaggle/input/datasets/parthupadhyay89/assignment-3/multimodal_train.tsv", sep="\t")
df_multimodel_validate_public = pd.read_csv("/kaggle/input/datasets/parthupadhyay89/assignment-3/multimodal_validate.tsv", sep="\t")

In [8]:
df_train = pd.concat([df_all_train_public, df_multimodel_train_public], ignore_index=True)
df_test = pd.concat([df_all_test_public, df_multimodel_test_public], ignore_index=True)
df_validate = pd.concat([df_all_validate_public, df_multimodel_validate_public], ignore_index=True)
df=pd.concat([df_train,df_test,df_validate],ignore_index=True)
df = df[['clean_title', '2_way_label', 'id']]
print("posts in original dataset :" ,len(df))

posts in original dataset : 1745767


In [9]:
df = df[['clean_title', '2_way_label', 'id']].dropna()
fake = df[df['2_way_label'] == 1]
non_fake = df[df['2_way_label'] == 0]
n_samples = 2500
fake_sampled = fake.sample(n=n_samples, random_state=42)
non_fake_sampled = non_fake.sample(n=n_samples, random_state=42)
df = pd.concat([fake_sampled, non_fake_sampled]).sample(frac=1, random_state=42).reset_index(drop=True)

print("posts in my dataset :" ,len(df))
print("fake post in my dataset :" ,len(fake_sampled))
print("non fake post in my dataset :" ,len(non_fake_sampled))
df.head()

posts in my dataset : 5000
fake post in my dataset : 2500
non fake post in my dataset : 2500


,clean_title,2_way_label,id
0,the way my binder rings open,1,cvolg1
1,hating point a political cartoon satirising th...,0,4ofyrj
2,i was just hanging out in the spc kicking it w...,0,clq0wm1
3,a cloud circling a cloud,1,9bktsq
4,how friends saved a woman waiting for a heart ...,1,6m0rnb


In [10]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['2_way_label']
)

print("Train size:", len(train_df))
print("distribution in training dataset : ",train_df['2_way_label'].value_counts())
print("Test size:", len(test_df))
print("distribution in test dataset : ",test_df['2_way_label'].value_counts())


Train size: 4000
distribution in training dataset :  2_way_label
1    2000
0    2000
Name: count, dtype: int64
Test size: 1000
distribution in test dataset :  2_way_label
0    500
1    500
Name: count, dtype: int64


In [11]:
MODEL_NAME = "bert-base-uncased"

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
MAX_LEN = 128

def tokenize_data(texts):
    return tokenizer(
        texts.tolist(),
        padding='max_length',
        truncation=True,
        max_length=MAX_LEN,
        return_tensors='pt'
    )

In [13]:
class FakeNewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [14]:
train_enc = tokenize_data(train_df['clean_title'])
test_enc = tokenize_data(test_df['clean_title'])

train_dataset = FakeNewsDataset(train_enc, train_df['2_way_label'])
test_dataset = FakeNewsDataset(test_enc, test_df['2_way_label'])

In [15]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,              
    per_device_train_batch_size=16,  
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",   
    logging_steps=50,           
)

In [16]:
from transformers import TrainerCallback

epoch_logs = []

class EpochMetricsCallback(TrainerCallback):
    def __init__(self, trainer):   
        self.trainer = trainer
    def on_epoch_end(self, args, state, control, **kwargs):
        train_metrics = trainer.evaluate(trainer.train_dataset)
        val_metrics = trainer.evaluate(trainer.eval_dataset)
        log = {
            "epoch": int(state.epoch),
            "train_accuracy": train_metrics['eval_accuracy'],
            "train_f1": train_metrics['eval_f1'],
            "train_precision": train_metrics['eval_precision'],
            "train_recall": train_metrics['eval_recall'],

            "val_accuracy": val_metrics['eval_accuracy'],
            "val_f1": val_metrics['eval_f1'],
            "val_precision": val_metrics['eval_precision'],
            "val_recall": val_metrics['eval_recall'],
        }

        epoch_logs.append(log)

        print("\n Epoch", int(state.epoch))
        print(log)

In [17]:
def compute_metrics(pred):
    logits, labels = pred
    preds = logits.argmax(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)
trainer.add_callback(EpochMetricsCallback(trainer))

trainer.train()
trainer.save_model("./results")  


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.193273,0.895391,0.805000,0.819277,0.763385,0.884000
2,0.796470,0.837949,0.806000,0.819030,0.767483,0.878000
3,0.595006,0.870802,0.817000,0.821114,0.803059,0.840000


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



 Epoch 1
{'epoch': 1, 'train_accuracy': 0.82375, 'train_f1': 0.8345458812485332, 'train_precision': 0.7863777089783281, 'train_recall': 0.889, 'val_accuracy': 0.805, 'val_f1': 0.8192771084337349, 'val_precision': 0.7633851468048359, 'val_recall': 0.884}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



 Epoch 3
{'epoch': 3, 'train_accuracy': 0.92575, 'train_f1': 0.9265396982438783, 'train_precision': 0.916789035731767, 'train_recall': 0.9365, 'val_accuracy': 0.817, 'val_f1': 0.8211143695014663, 'val_precision': 0.8030592734225621, 'val_recall': 0.84}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [18]:
results = trainer.evaluate()
print(results)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.8708015084266663, 'eval_accuracy': 0.817, 'eval_f1': 0.8211143695014663, 'eval_precision': 0.8030592734225621, 'eval_recall': 0.84, 'eval_runtime': 4.85, 'eval_samples_per_second': 206.185, 'eval_steps_per_second': 6.598, 'epoch': 3.0}


In [19]:
predictions = trainer.predict(test_dataset)
y_pred = predictions.predictions.argmax(axis=1)
y_true = predictions.label_ids

print(classification_report(y_true, y_pred))

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


              precision    recall  f1-score   support

           0       0.83      0.79      0.81       500
           1       0.80      0.84      0.82       500

    accuracy                           0.82      1000
   macro avg       0.82      0.82      0.82      1000
weighted avg       0.82      0.82      0.82      1000

Confusion Matrix:
 [[397 103]
 [ 80 420]]


In [20]:
model = BertForSequenceClassification.from_pretrained("./results")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [25]:
fake_examples = test_df[test_df['2_way_label'] == 1]['clean_title'].iloc[:3].tolist()
real_examples = test_df[test_df['2_way_label'] == 0]['clean_title'].iloc[:3].tolist()

texts = fake_examples + real_examples

encodings = tokenizer(
    texts,
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

input_ids = encodings['input_ids'].to(device)
attention_mask = encodings['attention_mask'].to(device)

token_type_ids = encodings.get('token_type_ids')
if token_type_ids is not None:
    token_type_ids = token_type_ids.to(device)

In [26]:
def forward_func(inputs_embeds, attention_mask=None):
    outputs = model(
        inputs_embeds=inputs_embeds,
        attention_mask=attention_mask
    )
    return outputs.logits

In [27]:
!pip install captum

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 72.9 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.

In [28]:
from captum.attr import LayerIntegratedGradients

embedding_layer = model.bert.embeddings

lig = LayerIntegratedGradients(forward_func, embedding_layer)

In [29]:
inputs_embeds = embedding_layer(input_ids)

baseline = torch.zeros_like(inputs_embeds).to(device)

In [30]:
all_attributions = []

for i in range(len(texts)):
    input_embed = inputs_embeds[i].unsqueeze(0)
    mask = attention_mask[i].unsqueeze(0)

    with torch.no_grad():
        pred = model(input_ids=input_ids[i].unsqueeze(0)).logits.argmax(dim=1).item()

    attributions, delta = lig.attribute(
        inputs=input_embed,
        baselines=baseline[i].unsqueeze(0),
        additional_forward_args=(mask,),
        target=pred,
        n_steps=50,
        return_convergence_delta=True
    )

    all_attributions.append((attributions, delta, pred))

In [31]:
import numpy as np

def process_attributions(attributions):
    scores = attributions.sum(dim=-1).squeeze(0)
    scores = scores / torch.norm(scores)
    return scores.cpu().detach().numpy()

In [32]:
for i, text in enumerate(texts):
    attributions, delta, pred = all_attributions[i]

    scores = process_attributions(attributions)
    tokens = tokenizer.convert_ids_to_tokens(input_ids[i])
    valid = [
        (tok, score) for tok, score in zip(tokens, scores)
        if tok not in ['[CLS]', '[SEP]', '[PAD]']
    ]
    valid_sorted = sorted(valid, key=lambda x: abs(x[1]), reverse=True)
    print(f"\nText {i+1}: {text}")
    print(f"Predicted class: {pred}")
    print(f"Convergence delta: {delta.item()}")
    print("Top-5 important tokens:")
    for tok, score in valid_sorted[:5]:
        print(f"{tok} ({score:.4f})")


Text 1: jessica long meets biological parents in russia in heartwarming reunion
Predicted class: 0
Convergence delta: 4.607687950134277
Top-5 important tokens:
reunion (-0.1859)
##war (0.1373)
jessica (-0.1275)
meets (-0.1051)
long (-0.0919)

Text 2: michael jacksons ghost tells lionel richies exwife he accidentally killed himself hears court
Predicted class: 0
Convergence delta: 3.8079144954681396
Top-5 important tokens:
tells (-0.1424)
accidentally (-0.1331)
ex (-0.0888)
ghost (-0.0767)
michael (-0.0560)

Text 3: my steak looks like the millennium falcon
Predicted class: 0
Convergence delta: 0.21279436349868774
Top-5 important tokens:
steak (-0.7237)
millennium (-0.2463)
falcon (-0.2016)
the (0.0918)
like (0.0657)

Text 4: official photo of the hong kong protests from the chinese government
Predicted class: 0
Convergence delta: -0.5505692958831787
Top-5 important tokens:
photo (-0.5460)
government (-0.3547)
of (-0.3268)
official (-0.2009)
protests (-0.1844)

Text 5: trump hails aret